## 1. Load and inspect

In [1]:
import pandas as pd

df = pd.read_csv("netflix_titles.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [ ]:
df.describe(include="all")

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
count,8807,8807,8807,6173,7982,7976,8797,8807.000000,8800,8807,8807,8807
unique,8807,2,8807,4528,7692,748,1767,NaN,14,220,514,8775
top,s1,Movie,Dick Johnson Is Dead,Rajiv Chilaka,David Attenborough,United States,"January 1, 2020",NaN,TV-MA,1 Season,"Dramas, International Movies","Paranormal activity at a lush, abandoned prope..."
freq,1,6131,1,19,19,2818,109,NaN,3207,1793,362,4
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014.180198,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.819312,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1925.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019.000000,NaN,NaN,NaN,NaN


In [19]:
df.isna().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64

In [17]:
# rows where duration is missing
df[df["duration"].isna()][["show_id", "title", "rating", "duration"]]

,show_id,title,rating,duration


## 2. Handle missing values, column by column

### `rating` and `duration` — fix the misaligned rows first

In [20]:
misaligned = df["rating"].str.contains(r"^\d+ min$", na=False)
df.loc[misaligned, "duration"] = df.loc[misaligned, "rating"]
df.loc[misaligned, "rating"] = pd.NA

print(f"Rows fixed: {misaligned.sum()}")
print(f"duration still missing: {df['duration'].isna().sum()}")
print(f"rating still missing: {df['rating'].isna().sum()}")

Rows fixed: 0
duration still missing: 0
rating still missing: 7


### `director`, `cast`, `country` — categorical text, missing means "not recorded"

Dropping rows isn't a good option here — `director` alone is missing for
about 30% of titles, so that would remove too much data. Filling with
`"Unknown"` keeps the rows and makes the gap explicit in any later
grouping or filtering.

In [21]:
df["director"] = df["director"].fillna("Unknown")
df["cast"] = df["cast"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")

### `rating` — remaining missing value

After fixing the misaligned rows, any still-missing rating gets a
`"Not Rated"` label rather than "Unknown", since that matches the existing
`"NR"`/`"UR"` categories already present in the data.

In [22]:
df["rating"] = df["rating"].fillna("Not Rated")

### `date_added` — leave missing values as missing

A date can't be reasonably guessed or filled with a placeholder without
being misleading, so the 10 rows with no `date_added` are left as actual
nulls (`NaT`) after parsing, rather than invented.

## 3. Fix the mixed-type `duration` column

`duration` stores two different units as text in the same column —
`"90 min"` for movies and `"2 Seasons"` for TV shows — which makes it
unusable for any numeric calculation as-is. Splitting it into two
numeric columns keeps both pieces of information and makes them usable:

In [23]:
df["duration_minutes"] = (
    df["duration"].str.extract(r"(\d+)\s*min")[0].astype("Float64")
)
df["duration_seasons"] = (
    df["duration"].str.extract(r"(\d+)\s*Season")[0].astype("Float64")
)

df[["type", "duration", "duration_minutes", "duration_seasons"]].head()

,type,duration,duration_minutes,duration_seasons
0,Movie,90 min,90.0,<NA>
1,TV Show,2 Seasons,<NA>,2.0
2,TV Show,1 Season,<NA>,1.0
3,TV Show,1 Season,<NA>,1.0
4,TV Show,2 Seasons,<NA>,2.0


## 4. Parse `date_added` into a real `datetime`

In [24]:
# Some values have stray leading/trailing whitespace, e.g. " September 25, 2021"
df["date_added"] = df["date_added"].str.strip()
df["date_added"] = pd.to_datetime(df["date_added"], format="%B %d, %Y", errors="coerce")

df["date_added"].dtype

dtype('<M8[us]')

## Final check

In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   show_id           8807 non-null   str           
 1   type              8807 non-null   str           
 2   title             8807 non-null   str           
 3   director          8807 non-null   str           
 4   cast              8807 non-null   str           
 5   country           8807 non-null   str           
 6   date_added        8797 non-null   datetime64[us]
 7   release_year      8807 non-null   int64         
 8   rating            8807 non-null   str           
 9   duration          8807 non-null   str           
 10  listed_in         8807 non-null   str           
 11  description       8807 non-null   str           
 12  duration_minutes  6131 non-null   Float64       
 13  duration_seasons  2676 non-null   Float64       
dtypes: Float64(2), datetime64[us](1), i

In [26]:
df.isna().sum()

show_id                0
type                   0
title                  0
director               0
cast                   0
country                0
date_added            10
release_year           0
rating                 0
duration               0
listed_in              0
description            0
duration_minutes    2676
duration_seasons    6131
dtype: int64

## 5. Export the cleaned DataFrame

In [27]:
df.to_csv("clean_netflix_titles.csv", index=False)
print("Saved clean_netflix_titles.csv —", df.shape)

Saved clean_netflix_titles.csv — (8807, 14)
